In [0]:
silverTablName = "saleslake_dev.silver_dev.cleanedsales"
goldTablName = "saleslake_dev.gold_dev.dim_sales"

# Create the target table if it doesn't exist
spark.sql(f"""
CREATE OR REPLACE TABLE {goldTablName} (
    sale_id INT,
    product STRING,
    category STRING,
    quantity INT,
    price DOUBLE,
    sale_date DATE,
    region STRING,
    is_active STRING,
    rec_version INT,
    start_effective_ts TIMESTAMP,
    end_effective_ts TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
MERGE INTO {goldTablName} tgt
USING (
    WITH latest_inv_silver AS (
        -- Step 1: Filter only new/updated silver records since last gold load
        SELECT *
        FROM {silverTablName}
        WHERE ingest_ts > (
            SELECT COALESCE(MAX(start_effective_ts), TO_TIMESTAMP('1990-01-01', 'yyyy-MM-dd'))
            FROM {goldTablName}
        )
    ),

    latest_rm_dup_silver AS (
        -- Step 2: Deduplicate — keep latest record per sale_id
        SELECT * FROM (
            SELECT *,
                   ROW_NUMBER() OVER (PARTITION BY sale_id ORDER BY ingest_ts DESC) AS rn
            FROM latest_inv_silver
        ) WHERE rn = 1
    ),

    silver_gold_rec AS (
        -- Step 3: Join with active gold records and classify each record
        SELECT
            s.*,

            g.is_active            AS is_active,
            g.rec_version          AS rec_version,
            g.start_effective_ts   AS start_effective_ts,
            g.end_effective_ts     AS end_effective_ts,
            CASE
                WHEN g.sale_id IS NULL THEN 1
                ELSE g.rec_version + 1
            END AS new_rec_version,
            CASE
                WHEN g.sale_id IS NULL THEN 'NEW'
                WHEN NOT (s.product   <=> g.product)
                  OR NOT (s.category  <=> g.category)
                  OR NOT (s.quantity  <=> g.quantity)
                  OR NOT (s.price     <=> g.price)
                  OR NOT (s.sale_date <=> g.sale_date)
                  OR NOT (s.region    <=> g.region)
                THEN 'CHANGE'
                ELSE 'NO_CHANGE'
            END AS rec_flag
        FROM latest_rm_dup_silver s
        LEFT JOIN (SELECT * FROM {goldTablName} WHERE is_active = 'Y') g
               ON s.sale_id = g.sale_id
    ),

    insert_flag AS (
        -- Step 4a: New version row to INSERT (for both NEW and CHANGE records)
        SELECT
            NULL    AS sale_merge_key,
            sale_id, product, category, quantity, price,
            sale_date, region,
            is_active, rec_version,
            start_effective_ts, end_effective_ts,
            new_rec_version, rec_flag,
            'INSERT' AS merge_flag
        FROM silver_gold_rec
        WHERE rec_flag IN ('NEW', 'CHANGE')
    ),

    changed_flag AS (
        -- Step 4b: Existing active row to CLOSE (UPDATE) for CHANGE records only
        SELECT
            sale_id AS sale_merge_key,
            sale_id, product, category, quantity, price,
            sale_date, region,
            is_active, rec_version,
            start_effective_ts, end_effective_ts,
            new_rec_version, rec_flag,
            'UPDATE' AS merge_flag
        FROM silver_gold_rec
        WHERE rec_flag = 'CHANGE'
    )

    -- Step 5: Combine both INSERT and UPDATE streams
    SELECT * FROM changed_flag
    UNION ALL
    SELECT * FROM insert_flag

) src
ON tgt.sale_id = src.sale_merge_key AND tgt.is_active = 'Y'

-- Close the old active record
WHEN MATCHED AND src.merge_flag = 'UPDATE' THEN UPDATE SET
    tgt.is_active        = 'N',
    tgt.end_effective_ts = CURRENT_TIMESTAMP()

-- Insert new record (new entry or new version of changed record)
WHEN NOT MATCHED AND src.merge_flag = 'INSERT' THEN INSERT (
    sale_id, product, category, quantity, price,
    sale_date, region,
    is_active, rec_version,
    start_effective_ts, end_effective_ts
)
VALUES (
    src.sale_id, src.product, src.category, src.quantity, src.price,
    src.sale_date, src.region,
    'Y', src.new_rec_version,
    CURRENT_TIMESTAMP(), TO_TIMESTAMP('9999-12-31', 'yyyy-MM-dd')
)
""")